# تمرین‌های Embedding و Reranker — فروشگاه نارنج
راهنمای هر تمرین در PDF است. **سلول اول** داده‌ها را می‌سازد؛ بقیه‌ی سلول‌ها همان کدهای PDF‌اند. جاهای `...` و `TODO` را پر کنید.

In [1]:
# Data cell: run this first (do not edit)
CORPUS = {
    "P01": "مشتری می‌تواند تا ۷ روز پس از تحویل، درخواست مرجوعی ثبت کند؛ به شرط اینکه کالا استفاده نشده و در بسته‌بندی اصلی باشد.",
    "P02": "اگر کالای سالم را فقط به دلیل انصراف پس بدهید، هزینه‌ی ارسال مرجوعی با شماست؛ اگر کالا معیوب باشد، این هزینه را فروشگاه می‌پردازد.",
    "P03": "پس از تأیید مرجوعی، مبلغ ظرف ۳ تا ۵ روز کاری به همان کارتی که با آن پرداخت کرده‌اید برگردانده می‌شود.",
    "P04": "کالاهای دیجیتال ۱۸ ماه گارانتی دارند. خرابی ناشی از ضربه یا آب‌خوردگی شامل گارانتی نمی‌شود.",
    "P05": "سفارش‌های تهران ۱ تا ۲ روز کاری و سفارش‌های شهرهای دیگر ۳ تا ۵ روز کاری بعد از ثبت تحویل داده می‌شوند.",
    "P06": "ارسال برای سفارش‌های بالای ۵۰۰ هزار تومان رایگان است.",
    "P07": "تا وقتی وضعیت سفارش به «ارسال شد» نرسیده، می‌توانید آن را از بخش «سفارش‌های من» لغو کنید.",
    "P08": "تغییر آدرس فقط قبل از ارسال سفارش و با تماس با پشتیبانی ممکن است.",
    "P09": "در هر سفارش فقط یک کد تخفیف قابل استفاده است و کد تخفیف شامل هزینه‌ی ارسال نمی‌شود.",
    "P10": "سفارش‌های بالای ۲ میلیون تومان را می‌توان در ۴ قسط ماهانه پرداخت کرد.",
    "P11": "اگر بسته هنگام تحویل آسیب دیده است، آن را تحویل نگیرید و از مأمور ارسال بخواهید صورت‌جلسه تنظیم کند.",
    "P12": "مجله‌ی نارنج هر هفته مقاله‌هایی درباره‌ی آشپزی، سفر و کتاب منتشر می‌کند.",
    "P13": "سفارشی که هنوز ارسال نشده، بدون هیچ هزینه‌ای قابل لغو است و مبلغ آن کامل برگردانده می‌شود.",
}

TOPIC = {"P01": "returns", "P02": "returns", "P03": "returns", "P04": "warranty",
         "P05": "shipping", "P06": "shipping", "P07": "orders", "P08": "orders",
         "P09": "payment", "P10": "payment", "P11": "shipping", "P12": "magazine",
         "P13": "orders"}

QUERIES = {
    "q1": "کفشی که خریدم اندازه‌م نیست؛ تا چند روز وقت دارم پسش بدم؟",
    "q2": "اگه گوشیم بیفته توی آب، گارانتی تعمیرش می‌کنه؟",
    "q3": "پولم کی برمی‌گرده؟",
    "q4": "سفارشم هنوز ارسال نشده؛ می‌تونم کنسلش کنم؟",
    "q5": "اگه کالای سالم رو پس بدم، هزینه‌ی پستش با کیه؟",
    "q6": "مهلت مرجوع کردن کالا چند روزه؟",
    "q7": "بسته‌م موقع تحویل له شده بود؛ باید چی‌کار کنم؟",
}

# reviewed gold: ALL passages that answer each query
GOLD = {"q1": ["P01"], "q2": ["P04"], "q3": ["P03"], "q4": ["P07", "P13"],
        "q5": ["P02"], "q6": ["P01"], "q7": ["P11"]}

# Ex1 — training records prepared by a teammate (one of them hides a false negative)
RECORDS = [
    {"query_id": "q1", "positive": "P01", "negatives": ["P12", "P02", "P06"]},
    {"query_id": "q4", "positive": "P07", "negatives": ["P12", "P08", "P13"]},
    {"query_id": "q2", "positive": "P04", "negatives": ["P09", "P11"]},
]

# Ex2 — toy 3-d vectors (axes: returns/deadline, money/cost, magazine/content)
TOY = {"q1": [0.8, 0.5, 0.1], "P01": [0.9, 0.4, 0.0],
       "P02": [0.6, 0.7, 0.2], "P12": [0.1, 0.2, 0.9]}

# Ex3 — InfoNCE scenarios: (positive score, negative scores, tau)
SCENARIOS = {
    "A_class_example": (0.65, [0.60, 0.20], 0.1),
    "B_easy_negatives_only": (0.65, [0.10, 0.20], 0.1),
    "C_high_temperature": (0.65, [0.60, 0.20], 1.0),
    "D_after_training": (0.80, [0.40, 0.20], 0.1),
    "E_hard_negative_wins": (0.55, [0.70, 0.20], 0.1),
}

# Ex4 — in-batch similarity matrix: rows = queries, columns = positives of the batch
BATCH_QUERIES = ["q1", "q3", "q4", "q6"]
BATCH_PASSAGES = ["P01", "P03", "P07", "P01"]
SIM = [
    [0.82, 0.41, 0.15, 0.82],
    [0.45, 0.78, 0.20, 0.45],
    [0.18, 0.22, 0.85, 0.18],
    [0.80, 0.40, 0.12, 0.80],
]

# Ex5 — reranker scores and graded labels for q5
RERANK_Q5 = [
    {"pid": "P02", "relevance": 2, "score": 2.1},
    {"pid": "P01", "relevance": 1, "score": 2.4},
    {"pid": "P05", "relevance": 0, "score": 0.3},
    {"pid": "P12", "relevance": 0, "score": -1.5},
]

# Ex6 — a controlled run of the funnel
DENSE_TOP5 = {
    "q1": ["P02", "P01", "P03", "P11", "P06"],
    "q3": ["P10", "P06", "P09", "P02", "P05"],
    "q4": ["P08", "P13", "P07", "P05", "P11"],
    "q2": ["P04", "P11", "P01", "P05", "P12"],
    "q7": ["P02", "P05", "P08", "P01", "P11"],
}
RERANKED_TOP3 = {
    "q1": ["P01", "P02", "P03"],
    "q3": ["P02", "P10", "P06"],
    "q4": ["P07", "P13", "P08"],
    "q2": ["P04", "P11", "P01"],
    "q7": ["P11", "P02", "P01"],
}

print(len(CORPUS), "passages |", len(QUERIES), "queries")

13 passages | 7 queries


## بخش ۰ — آشنایی با داده (۵ دقیقه)

## تمرین ۱ — مثبت، منفی آسان، منفی سخت، منفی کاذب (۱۵ دقیقه)

### ۱-الف) اول روی کاغذ

> R1: P12 = E N , P2 = HN , P6 = EN

> R2: P12 = EN , P8 = HN , P13 = FN

> R3: P9 = EN , P11 = EN

### ۱-ب) کد بزنید

In [3]:
def classify_negatives(record):
    q = record["query_id"]
    pos_topic = TOPIC[record["positive"]]
    out = {}
    for pid in record["negatives"]:
        if pid in GOLD[q]:
            out[pid] = "FALSE NEGATIVE"    # TODO 1: pid is in GOLD of this query

        elif TOPIC[pid] == pos_topic:                    # TODO 2: same topic as the positive
            out[pid] = "hard"
        else:
            out[pid] = "easy"
    return out

for r in RECORDS:
    print(r["query_id"], classify_negatives(r))

q1 {'P12': 'easy', 'P02': 'hard', 'P06': 'easy'}
q4 {'P12': 'easy', 'P08': 'hard', 'P13': 'FALSE NEGATIVE'}
q2 {'P09': 'easy', 'P11': 'easy'}


> R2: مدل رو مجبور میکنیم ( لاس ) که نمونه ای که در حقیقت مثبت است را از سوال دور کند

> متن بسیار فریبنده و مشابه است با وجودی که تاپیک فرق دارد متن می تواند شباهت بزرگی تولید کند

>بله منفی و مثبت بودن در اسکوپ سراسری دیتاست ست معنا ندارد و رابطه هر چانک با هر سوال این علامت را مشخص میکند

## تمرین ۲ — شباهت کسینوسی: نقشه‌ی معنا با دست (۱۰ دقیقه)

In [5]:
import math

def dot(a, b):
    return sum(x * y for x, y in zip(a, b))  # TODO: sum of a[i]*b[i]

def norm(a):
    return math.sqrt(dot(a,a))                # TODO: math.sqrt of dot(a, a)

def cosine(a, b):
    return dot(a, b) / (norm(a)*norm(b) )    # TODO

for pid in ("P01", "P02", "P12"):
    print(pid, round(cosine(TOY["q1"], TOY[pid]), 3))

P01 0.985
P02 0.95
P12 0.307


## تمرین ۳ — loss کنتراستیو InfoNCE (۲۵ دقیقه)

```
p    = exp(s_pos/τ) / ( exp(s_pos/τ) + Σ exp(s_neg/τ) )
loss = -log(p)
```

### ۳-الف) اول روی کاغذ (با ماشین‌حساب)

### ۳-ب) کد بزنید

In [6]:
def info_nce(pos, negs, tau):
    top = math.exp(pos/ tau)                        # TODO 1: math.exp(pos / tau)
    bottom = top + sum(math.exp(n/tau) for n in negs)                  # TODO 2: sum of exp(n / tau) for every n in negs
    p = top / bottom
    return p, - math.log(p)                   # TODO 3: the loss

for name, (pos, negs, tau) in SCENARIOS.items():
    p, loss = info_nce(pos, negs, tau)
    print(f"{name:24} p={p:.3f} loss={loss:.3f}")

A_class_example          p=0.618 loss=0.481
B_easy_negatives_only    p=0.985 loss=0.015
C_high_temperature       p=0.386 loss=0.951
D_after_training         p=0.980 loss=0.021
E_hard_negative_wins     p=0.181 loss=1.707


<div dir="rtl">

> منفی های ساده مشخص است فاصله و بی ارتباطی زیادی دارند لزا لاس تولید شده کوچک است

> exp(0.6) بیشترین سهم را دارد چرا که باعث بزرگ شدن لاس میشود که ناشی از کوچک شدن کسر میباشد

>چون تا مخرج توان اکسپوننشیال میباشد اگر تا کوچک باشد باعث تیز شدن توزیع احتمال شده و باعث میشود امتیاز مثبت و منفی بزرگ تر شود و لاس شدید تر اعمال شود که باعث افزایش ریسک اورفیت میشود ولی تا بزرگ توزیع رو پخش تر میکند و باعث میشود ریسک آندرفیت افزایش یابد

> امتیاز متون مثبت را بالا تر میبرد و کوساین سیمیلاریتی آنها به 1 میل میکند ولی متن های منفی به -1 میل میکند

## تمرین ۴ — منفی‌های درون بچ و خطر منفی کاذب (۲۰ دقیقه)

### ۴-الف) loss بچ را حساب کنید

In [7]:
def batch_loss(sim, tau=0.1, mask_duplicates=False):
    losses = []
    for i in range(len(sim)):
        pos = sim[i][i]                                  # TODO 1
        negs = []
        for j in range(len(sim[i])):
            if j == i:
                continue
            if mask_duplicates and (BATCH_PASSAGES[j] == BATCH_PASSAGES[i]):            # TODO 3 (part b)
                continue
            negs.append(sim[i][j])                       # TODO 2
        losses.append(info_nce(pos, negs, tau)[1])
    return losses

before = batch_loss(SIM)
print([round(x, 3) for x in before], "mean:", round(sum(before) / len(before), 3))

[0.702, 0.074, 0.004, 0.703] mean: 0.371


> چون 01 دوبار داخل پسیج ها وجود داره فقط کافیه یکبار بیاریمش ، اگر دو بار بیاریمش مدل ناچار میشه یک بار منفی در نظر بگیردش و یه بار مثبت پس لاس تولید میشه برای اون نسخه ای که منفی در نظر گرفته برای همین در این بخش از کد

            if mask_duplicates and (BATCH_PASSAGES[j] == BATCH_PASSAGES[i]):            # TODO 3 (part b)
تصمیم گرفتیم پسیج های تکراری رو رد کنیم تا هر پسیج یکبار تو محاسبه بیاد

> باعث میشود پسیج 01 که منفی میشود مدل را به سمت دور شدن از پاسخ طلایی سوق دهد برای همین داپلیکیت را ماسک میکنیم

> ماسک کردن داپلیکیت
>


### ۴-ب) منفی کاذب را حذف کنید

In [8]:
after = batch_loss(SIM, mask_duplicates=True)
print([round(x, 3) for x in after], "mean:", round(sum(after) / len(after), 3))

[0.018, 0.074, 0.004, 0.019] mean: 0.029


> چون پسیج یکسان پاسخ هر دو بود ، باعث میشد یکی را منفی در نظر بگیرد

<div dir="rtl">

## تمرین ۵ — loss زوجی reranker و برچسب‌های درجه‌بندی‌شده (۲۰ دقیقه)

```
L = log(1 + exp(-(s_pos - s_neg)))
```

### ۵-الف) تابع و مثال‌های کلاس

In [9]:
def pairwise_loss(s_pos, s_neg):
    return  math.log(1 + math.exp(-(s_pos - s_neg)))    # TODO

print(round(pairwise_loss(1.2, 0.7), 3), round(pairwise_loss(0.4, 1.1), 3))

0.474 1.103


### ۵-ب) از برچسب درجه‌بندی‌شده به جفت‌ها

In [10]:
pairs = []
for a in RERANK_Q5:
    for b in RERANK_Q5:
        if a["relevance"] > b["relevance"]:                                         # TODO 1
            loss = pairwise_loss(a["score"], b["score"])
            pairs.append((a["pid"], b["pid"], loss))

for pa, pb, loss in pairs:
    print(pa, ">", pb, round(loss, 3))
print("pairs:", len(pairs), "mean:", round(sum(p[2] for p in pairs) / len(pairs)))        # TODO 2

P02 > P01 0.854
P02 > P05 0.153
P02 > P12 0.027
P01 > P05 0.116
P01 > P12 0.02
pairs: 5 mean: 0


## تمرین ۶ — قیف بازیابی: قبل و بعد از reranker (۱۵ دقیقه)

In [ ]:
def recall_at_k(ranked, gold, k):
    return ...                                   # TODO 1

def rr_at_k(ranked, gold, k):
    for rank, p in enumerate(ranked[:k], start=1):
        ...                                      # TODO 2
    return 0

rows = []
for q in DENSE_TOP5:
    g = GOLD[q]
    r = (recall_at_k(DENSE_TOP5[q], g, 5),     # dense R@5
         recall_at_k(DENSE_TOP5[q], g, 3),     # dense R@3
         recall_at_k(RERANKED_TOP3[q], g, 3),  # reranked R@3
         rr_at_k(DENSE_TOP5[q], g, 3),         # dense RR@3
         rr_at_k(RERANKED_TOP3[q], g, 3))      # reranked RR@3
    rows.append(r)
    print(q, [round(x, 2) for x in r])
print("mean", [round(sum(col) / len(col), 3) for col in zip(*rows)])

## تمرین ۷ — آزمایشگاه: مدل‌های واقعی روی سؤال‌های فارسی (۳۰ دقیقه)

### قدم ۱ — نصب و import

In [ ]:
%pip install langchain-core langchain-openai langchain-community flashrank
# restart the kernel after install, then re-run the first (data) cell

In [ ]:
import os
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_compressors import FlashrankRerank
print("imports ok | key found:", bool(os.getenv("OPENROUTER_API_KEY")))

### قدم ۲ — ساختن Documentها (کد بزنید)

In [ ]:
documents = []
for pid, text in CORPUS.items():
    documents.append(...)          # TODO: a Document with page_content=text and metadata={"pid": pid}
print(len(documents), documents[0].metadata)

### قدم ۳ — نقشه‌کش: جستجوی dense (نیاز به کلید)

In [ ]:
embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    check_embedding_ctx_length=False,
)
vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(documents)

def dense_search(query, k=5):
    hits = vector_store.similarity_search_with_score(query, k=k)
    return [(doc.metadata["pid"], round(score, 3)) for doc, score in hits]

for qid in ["q1", "q3", "q4", "q2", "q7"]:
    print(qid, dense_search(QUERIES[qid]))

### قدم ۴ — قاضی: rerank با FlashRank (کد بزنید)

In [ ]:
reranker = FlashrankRerank(top_n=3)       # default: multilingual MultiBERT

def rerank(query, candidates):
    out = reranker.compress_documents(documents=candidates, query=query)
    return [(d.metadata["pid"], round(float(d.metadata["relevance_score"]), 3)) for d in out]

for qid in ["q1", "q3", "q4", "q2", "q7"]:
    q = QUERIES[qid]
    # WITH key: candidates = dense top-5 documents
    candidates = [doc for doc, score in vector_store.similarity_search_with_score(q, k=5)]
    # WITHOUT key: comment the line above and use all 13 documents instead:
    # candidates = documents
    print(qid, ...)                         # TODO: call rerank

### قدم ۵ — آزمایش زبان: reranker انگلیسی روی سؤال فارسی

In [ ]:
reranker_en = FlashrankRerank(top_n=3, model="ms-marco-TinyBERT-L-2-v2")
for qid in ["q1", "q4", "q7"]:
    q = QUERIES[qid]
    candidates = documents        # or dense top-5 if you have a key
    out = reranker_en.compress_documents(documents=candidates, query=q)
    print(qid, [d.metadata["pid"] for d in out])

### قدم ۶ — جدول نتایج

## تمرین ۸ — مقاله‌خوانی هدفمند (۱۵ دقیقه)

## تمرین ۹ — جستجو: بهترین encoder فارسی برای fine-tune (۱۵ دقیقه)

## سؤال‌های پایانی